<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/New-Baseline-v2/mnps_new_baseline%20v7.5.4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **MNPS Job Equity New Baseline 7.5.4**
> A notebook to help you get started  
> DSI DSSG + MNPS   

> # **Version 7.5.4 Changes**
> - **Alignment Fix**: Force `major_role_group` to match justification/title signals when inconsistent
> - **Principal vs Assistant Principal**: Rules based on experience, functions, and licensure (ILL-B or ILL-P)
> - **Minor Sub Rules**: For Teacher, Librarian, Counselor — only use `Lead`; otherwise leave blank
> - **Expanded Role Set**: Adds `Liaison`, `Representative`, `Facilitator`
> - **Role Disambiguations**: Clerk vs Administrative Assistant; Instructor vs Teacher; Assistant vs Coordinator; Rep vs Facilitator vs Coordinator vs Liaison
> - **Carries v7.5.3 fixes**: Rate limit backoff, Specialist discouraging, Coordinator/Coach/Manager refinement, executive Lead guard


In [ ]:
# ==== 1) Imports, paths, inputs from v7.1 artifacts ====
import os, json, shutil, datetime as dt, zipfile
from pathlib import Path
import pandas as pd
import numpy as np
import re
import time
import random
from google.colab import drive
from openai import OpenAI

# Mount Google Drive
drive.mount('/content/drive')

# Create unique run folder
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
run_folder = f"RUN_{timestamp}"
base_path = Path("/content/drive/My Drive/Colab Notebooks/Run Results")
run_path = base_path / run_folder
run_path.mkdir(parents=True, exist_ok=True)

# Create outputs subfolder
OUTPUTS_DIR = run_path / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Run folder: {run_path}")
print(f"📁 Outputs dir: {OUTPUTS_DIR}")

# Load all required files - Updated for Colab root path
RUN_ROOT = Path('/content')

# Unzip MNPS Prompt Resources if needed
ZIP_FILE = RUN_ROOT / "MNPS Prompt Resources.zip"
if ZIP_FILE.exists():
    print(f"📦 Found {ZIP_FILE}, extracting...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_ref.extractall(RUN_ROOT)
    print("✅ Extracted MNPS Prompt Resources")
else:
    print("⚠️  MNPS Prompt Resources.zip not found - make sure to upload it")

# Core data files
BATCH_INPUT_CSV = RUN_ROOT / "Sample JDs.csv"
GT_MASTERFILE_CSV = RUN_ROOT / "Ground Truth Masterfile.csv"

# MNPS Prompt Resources (from extracted zip)
MNPS_ROLES_CSV = RUN_ROOT / "MNPS Roles.csv"
MNPS_KSACS_CSV = RUN_ROOT / "MNPS KSACs.csv"
COMPETENCY_EXTENDED_CSV = RUN_ROOT / "Competency Extended Descriptions.csv"
KORN_FERRY_CSV = RUN_ROOT / "Korn_Ferry Lominger 38 Competencies.csv"

print(f"📄 Batch input: {BATCH_INPUT_CSV}")
print(f"📄 Ground truth: {GT_MASTERFILE_CSV}")
print(f"📄 MNPS roles: {MNPS_ROLES_CSV}")
print(f"📄 MNPS KSACs: {MNPS_KSACS_CSV}")
print(f"📄 Competency Extended: {COMPETENCY_EXTENDED_CSV}")
print(f"📄 Korn Ferry: {KORN_FERRY_CSV}")

# Load data
df = pd.read_csv(BATCH_INPUT_CSV, encoding='latin1')
gt_df = pd.read_csv(GT_MASTERFILE_CSV, encoding='latin1')
roles_df = pd.read_csv(MNPS_ROLES_CSV, encoding='latin1')
ksacs_df = pd.read_csv(MNPS_KSACS_CSV, encoding='latin1')
competency_df = pd.read_csv(COMPETENCY_EXTENDED_CSV, encoding='latin1')
korn_ferry_df = pd.read_csv(KORN_FERRY_CSV, encoding='latin1')

print(f"✅ Loaded {len(df)} job descriptions")
print(f"✅ Loaded {len(gt_df)} ground truth records")
print(f"✅ Loaded {len(roles_df)} MNPS roles")
print(f"✅ Loaded {len(ksacs_df)} MNPS KSACs")
print(f"✅ Loaded {len(competency_df)} competency descriptions")
print(f"✅ Loaded {len(korn_ferry_df)} Korn Ferry competencies")


In [ ]:
# ==== 2) Build attribute-only view (ignore title) ====

preds = df.copy()
ATTR_COLS = [
    'Position Summary', 'Essential Functions', 'Work Experience', 'Education',
    'Licenses and Certifications', 'Knowledge, Skills and Abilities'
]
attrs = df[ATTR_COLS].fillna('')
text = attrs['Position Summary'] + ' ' + attrs['Essential Functions'] + ' ' + \
       attrs['Work Experience'] + ' ' + attrs['Education'] + ' ' + \
       attrs['Licenses and Certifications'] + ' ' + attrs['Knowledge, Skills and Abilities']
print(f"✅ Built attribute-only view for {len(text)} JDs")


In [ ]:
# ==== 3) Role logic, refinements, and alignment enforcement ====

# Roles list
role_columns = [col for col in roles_df.columns if 'role' in col.lower()]
VALID_ROLES = roles_df[role_columns[0]].dropna().tolist() if role_columns else roles_df.iloc[:,0].dropna().tolist()

MAJOR_ALLOWED = set(VALID_ROLES + [
    'Liaison','Representative','Facilitator','Principal','Assistant Principal',
    'Administrative Assistant','Clerical Support','Instructor'
])

EXECUTIVE_ROLES = {'Coordinator','Principal','Director','Manager','Assistant Principal'}
MINOR_ALLOWED = {'','I','II','III','Lead'}

SPECIALIST_FALLBACKS = [
    ('Technician', 'technical|repair|maintenance|install|troubleshoot|equipment|hands-on|tools|machinery|systems'),
    ('Analyst', 'analy(s|z)e|research|evaluate|assess|statistical|quantitative|qualitative|metrics|reports'),
    ('Teacher', 'classroom|lesson|instruction|teacher|students|curriculum|teaching|academic'),
    ('Coach', 'coach|ment(or|ing)|professional development|co-teach|model lessons|plc'),
    ('Clerical Support', 'clerk|clerical|records|data entry|office support|administrative|filing|correspondence'),
    ('Counselor', 'counsel|social-emotional|guidance|therapy|mental health|behavioral'),
    ('Manager', 'manage|supervise|budget|oversight|lead team|program manager|policy|strategic|planning'),
    ('Accountant', 'accounting|financial|bookkeeping|audit|budget|finance|accounts payable|accounts receivable|fiscal'),
    ('Coordinator', 'coordinate|organize|facilitate|liaison|program|project|event'),
    ('Liaison', 'liaison|bridge|interface|connect families|community partners'),
    ('Representative', 'representative|rep\b|outreach|frontline engagement'),
    ('Facilitator', 'facilitate|facilitator|workshops|sessions')
]

def normalize_minor(minor: str, major: str) -> str:
    s = '' if pd.isna(minor) else str(minor).strip()
    s = s if s in MINOR_ALLOWED else 'I'
    # Teacher/Librarian/Counselor have blank unless Lead
    if major in {'Teacher','Librarian','Counselor'} and s != 'Lead':
        return ''
    # Executive roles rarely Lead
    if major in EXECUTIVE_ROLES and s == 'Lead':
        return 'III' if major in {'Director','Principal'} else 'II'
    return s

def discourage_specialist(job_text: str, proposed_major: str) -> str:
    if proposed_major != 'Specialist':
        return proposed_major
    t = (job_text or '').lower()
    for major, pattern in SPECIALIST_FALLBACKS:
        if re.search(pattern, t):
            return major
    return proposed_major

def refine_coordinator_coach_manager(job_text: str, proposed_major: str) -> str:
    if proposed_major not in {'Coordinator','Coach','Manager'}:
        return proposed_major
    t = (job_text or '').lower()
    if re.search(r'(instructional|mentor|professional development|co-teach|model lessons|plc)', t):
        return 'Coach'
    if re.search(r'(strategic|policy|budget|supervis|manage|oversight|planning)', t):
        return 'Manager'
    if re.search(r'(coordinate|organize|facilitate|liaison|program|project|event)', t):
        return 'Coordinator'
    return proposed_major

def principal_vs_ap(row: pd.Series, proposed_major: str) -> str:
    t_exp = str(row.get('Work Experience','')).lower()
    t_fn  = str(row.get('Essential Functions','')).lower()
    t_cert= str(row.get('Licenses and Certifications','')).lower()
    if ('5 full years of certificated experience' in t_exp and
        '2 years of leadership experience' in t_exp and
        'supervises staff' in t_fn and
        (('ill-b' in t_cert) or ('ill-p' in t_cert))):
        return 'Principal'
    if ('3 years of certificated experience' in t_exp and
        'assists with supervision of staff' in t_fn and
        (('ill-b' not in t_cert) and ('ill-p' not in t_cert))):
        return 'Assistant Principal'
    return proposed_major

def other_role_disambiguations(job_text: str, proposed_major: str) -> str:
    t = (job_text or '').lower()
    # Clerk vs Administrative Assistant
    if re.search(r'\bclerk\b', t):
        return 'Clerical Support'
    if re.search(r'administrative (assistant|support)|office support|calendar|correspondence', t):
        return 'Administrative Assistant'
    # Instructor vs Teacher
    if re.search(r'jrotc|military science|cte|driver education|behind the wheel', t):
        return 'Instructor'
    if re.search(r'certified teacher|classroom instruction|lesson plans', t):
        return 'Teacher'
    # Assistant vs Coordinator
    if proposed_major == 'Assistant' and re.search(r'coordinate|organize|facilitate|liaison|program|project', t):
        return 'Coordinator'
    # Rep vs Facilitator vs Coordinator vs Liaison
    if re.search(r'liaison', t):
        return 'Liaison'
    if re.search(r'facilitat(e|or)', t):
        return 'Facilitator'
    if re.search(r'\brepresentative\b|\brep\b', t):
        return 'Representative'
    return proposed_major

def align_with_justification_and_title(major: str, new_title: str, justification: str) -> str:
    # If both title and justification mention the same known role term, use it
    signals = ['Coordinator','Coach','Manager','Principal','Assistant Principal','Librarian','Counselor','Technician','Analyst','Accountant','Instructor','Liaison','Representative','Facilitator','Administrative Assistant','Clerical Support','Teacher']
    title_l = (new_title or '').lower()
    just_l  = (justification or '').lower()
    for sig in signals:
        s = sig.lower()
        if s in title_l and s in just_l:
            return sig
    return major

print('✅ Role logic and alignment helpers defined')


In [ ]:
# ==== 4) Build KSACs text (same as prior versions) ====

def build_ksacs_text():
    ksacs_text = "MNPS Knowledge, Skills, Abilities, and Competencies (KSACs):\n\n"
    for df_, role_col, text_col in [
        (ksacs_df, next((c for c in ksacs_df.columns if 'role' in c.lower()), None), next((c for c in ksacs_df.columns if 'ksac' in c.lower()), None))
    ]:
        if role_col and text_col:
            for _, r in df_.iterrows():
                role = r.get(role_col, '')
                ksacs = r.get(text_col, '')
                if role and ksacs:
                    ksacs_text += f"**{role}**:\n{ksacs}\n\n"
    return ksacs_text

KSACS_TEXT = build_ksacs_text()
print(f"✅ Built KSACs text length: {len(KSACS_TEXT)}")


In [ ]:
# ==== 5) Prompt (carrying v7.5.3 with updates) ====
zero_shot_prompt = \
""" Objective: Evaluate and group jobs based on job attributes (not titles).

Follow MNPS classification standards and these guidelines:
- Use Coordinator vs Coach vs Manager distinctions correctly
- Discourage overuse of Specialist
- Executive roles (Coordinator, Principal, Director, Manager) rarely have 'Lead'
- Teacher/Librarian/Counselor have no minor level unless truly 'Lead'
- Principal vs Assistant Principal rules per experience, functions, licensure
- Architect roles: Facility-Focused vs Technology-Focused

Return JSON:
{
  "new_job_title": "Descriptive title with major_role_group and minor_sub_group",
  "major_role_group": "One of MNPS roles",
  "minor_sub_group": "I, II, III, or Lead (or blank when appropriate)",
  "grouping_justification": "Detailed justification by attributes and KSACs"
}
"""
print('✅ Prompt defined')


In [ ]:
# ==== 6) OpenAI API with retry (from v7.5.3) ====
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
client = OpenAI()
MODEL_ID = "gpt-4o-2024-11-20"

def call_llm_json_with_retry(prompt: str, model: str = None, max_retries: int = 3) -> dict:
    if model is None:
        model = MODEL_ID
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[{"role":"user","content":prompt}],
                response_format={"type":"json_object"},
                temperature=0.2
            )
            return json.loads(resp.choices[0].message.content)
        except Exception as e:
            msg = str(e).lower()
            if ("429" in msg or "rate limit" in msg or "quota" in msg) and attempt < max_retries-1:
                wait = (2 ** attempt) + random.uniform(0,1)
                print(f"⚠️ Rate limit; waiting {wait:.1f}s before retry...")
                time.sleep(wait)
                continue
            raise
print('✅ OpenAI client and retry defined')


In [ ]:
# ==== 7) Batch Processing with alignment enforcement ====
from tqdm import tqdm

def process_job(row_idx: int, row: pd.Series) -> dict:
    job_text = f"""Position Summary: {row.get('Position Summary','')}
Essential Functions: {row.get('Essential Functions','')}
Work Experience: {row.get('Work Experience','')}
Education: {row.get('Education','')}
Licenses and Certifications: {row.get('Licenses and Certifications','')}
Knowledge, Skills and Abilities: {row.get('Knowledge, Skills and Abilities','')}"""

    prompt = f"""{zero_shot_prompt}

Available MNPS Roles: {', '.join(sorted(MAJOR_ALLOWED))}

{KSACS_TEXT}

Job Description to Classify:
{job_text}

Return your response as specified."""

    try:
        data = call_llm_json_with_retry(prompt, MODEL_ID)
        major = data.get('major_role_group','Other')
        minor = data.get('minor_sub_group','')
        new_title = data.get('new_job_title','')
        just = data.get('grouping_justification','')

        # Post-processing refinements
        major = discourage_specialist(job_text, major)
        major = refine_coordinator_coach_manager(job_text, major)
        major = principal_vs_ap(row, major)
        major = other_role_disambiguations(job_text, major)
        major = align_with_justification_and_title(major, new_title, just)

        minor = normalize_minor(minor, major)

        if not new_title:
            new_title = f"{major} {minor}".strip()

        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Title',''),
            'new_job_title': new_title,
            'major_role_group': major,
            'minor_sub_group': minor,
            'grouping_justification': just,
            'model_used': MODEL_ID
        }
    except Exception as e:
        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Title',''),
            'new_job_title': 'Error',
            'major_role_group': 'Other',
            'minor_sub_group': '',
            'grouping_justification': f'Error: {e}',
            'model_used': MODEL_ID
        }

# Run
results = []
for idx, r in tqdm(df.iterrows(), total=len(df), desc='Processing jobs'):
    results.append(process_job(idx, r))
    time.sleep(0.2)

results_df = pd.DataFrame(results)
output_path = OUTPUTS_DIR / 'Job_Classifications_Batch_gpt4o_v754.csv'
results_df.to_csv(output_path, index=False)
print(f"✅ Saved: {output_path}")


In [ ]:
# ==== 8) Summary and quality checks ====

preds = results_df.copy()
major_counts = preds['major_role_group'].value_counts()
minor_counts = preds['minor_sub_group'].value_counts()

summary = pd.DataFrame({
    'metric': ['rows','unique_major','unique_minor','specialist_count','exec_lead_count'],
    'value': [
        len(preds),
        len(major_counts),
        len(minor_counts),
        int((preds['major_role_group']=='Specialist').sum()),
        int((preds['major_role_group'].isin(list(EXECUTIVE_ROLES)) & (preds['minor_sub_group']=='Lead')).sum())
    ]
})
summary_path = OUTPUTS_DIR / 'summary_stats_gpt4o_v754.csv'
summary.to_csv(summary_path, index=False)

# Alignment issues: justification vs major role; title vs major role
issues = []
for _, r in preds.iterrows():
    major = str(r['major_role_group']).lower()
    jt = str(r['new_job_title']).lower()
    just = str(r['grouping_justification']).lower()
    if major != 'other':
        if major not in just or major not in jt:
            issues.append({
                'row': r['source_row_index'],
                'major_role_group': r['major_role_group'],
                'new_job_title': r['new_job_title'],
                'justification_excerpt': (r['grouping_justification'] or '')[:160]
            })

if issues:
    issues_df = pd.DataFrame(issues)
    issues_path = OUTPUTS_DIR / 'alignment_issues_gpt4o_v754.csv'
    issues_df.to_csv(issues_path, index=False)
    print(f"⚠️ Alignment issues: {len(issues)} → {issues_path}")
else:
    print('✅ No alignment issues')

print(f"✅ Summary saved: {summary_path}")
print(summary.to_string(index=False))
